# Masking data-cubes using geometry objects

In [ ]:
from earthkit.data.utils.testing import earthkit_remote_test_data_file

from earthkit import data as ekd
from earthkit import plots as ekp
from earthkit import transforms as ekt

## Load some test data

All `earthkit-transforms` methods can be called with `earthkit-data` objects or with
pre-loaded `xarray` or `geopandas` objects.

In this example we will use hourly ERA5 2m temperature data on a 0.5x0.5 spatial grid for the year 2015 as
our physical data; and we will use the NUTS geometries which are stored in a geojson file. As we are
going to repeat calls to the aggregation methods, we will convert the data to `xarray.Dataset` and `geopandas`
to avoid repeated conversion.

The cells below downloads some sample ERA5 data and opens as an xarray object (first cell), then plots the
data using `earthkit.plots`.

In [ ]:
# Get some demonstration ERA5 data, this could be any url or path to an ERA5 grib or netCDF file.
# remote_era5_file = earthkit_remote_test_data_file("era5_temperature_europe_2015.grib") # Large file
remote_era5_file = earthkit_remote_test_data_file("era5_temperature_europe_20150101.grib")
era5_data = ekd.from_source("url", remote_era5_file)

# Open as an xarray dataset, renaming the 2m temperature variable to something more manageable
era5_xr = era5_data.to_xarray(time_dim_mode="valid_time").rename({"2t": "t2m"})
era5_xr

In [ ]:
ekp.geo.plot(era5_xr.mean(dim="valid_time"), domain="Europe")

We do the same for our geojson data, except this time we open as a geopandas objects and plot using the
built in geopandas methods.

In [ ]:
# Use some demonstration polygons stored, this could be any url or path to geojson file
remote_nuts_url = earthkit_remote_test_data_file("NUTS_RG_60M_2021_4326_LEVL_0.geojson")
nuts_data = ekd.from_source("url", remote_nuts_url).to_geopandas()

nuts_data[:5]

In [ ]:
ekp.choropleth(nuts_data, z=None, labels="FID", domain="Europe").show()

## Mask data with the geometries

`shapes.mask` applies the features in the geometry object (`nuts_data`) to the data object (`era5_data`).
It returns an xarray object with an additional dimension, and coordinate variable, corresponding to the 
features in the geometry object.
By default this is the index of the input geodataframe, in this example the index is just an integer
count so it takes the default name `index`.

In [ ]:
masked_data = ekt.spatial.mask(era5_xr, nuts_data)
masked_data

It is possible to specify a column in the geodataframe to use for the new dimension, for example in NUTS the
`NAME_LATN`  contains the name of the country (using the Latin alphabet) for each feature. This will be useful
the subsequent plotting.

In [ ]:
masked_data = ekt.spatial.mask(era5_xr, nuts_data, mask_dim="NAME_LATN")
masked_data

We can now use `earthkit-plots` to plot the masked features, below we select the first time step and produce
for the first 12 features.

In [ ]:
plot_data = masked_data.isel(valid_time=0)

figure = ekp.Figure(domain="Europe", size=(9, 7), rows=3, columns=4)
nmaps = figure.rows * figure.columns

for i in range(nmaps):
    subplot = figure.add_map()
    subplot.pcolormesh(plot_data.isel(NAME_LATN=i), style="2m-temperature-spectral-celsius")
    subplot.title("{NAME_LATN}")

figure.title("{variable_name} - {valid_time:%Y-%m-%d}")
figure.coastlines()
figure.borders()
figure.legend()

figure.show()

### Union geometries

`ekt.spatial.mask` allows users to unify all geometries into a single feature when applying the mask via the
kwarg `union_geometries`.

In [ ]:
single_masked_data = ekt.spatial.mask(era5_xr, nuts_data, union_geometries=True)
single_masked_data

In [ ]:
ekp.geo.plot(single_masked_data.mean(dim="valid_time"), domain="Europe").show()